In [0]:
import csv
from typing import List, Dict

def read_csv_file(file_path: str) -> List[Dict[str, str]]:
    """
    Read CSV file and return list of dictionaries.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        List of dictionaries containing row data
    """
    data = []
    with open(file_path, 'r') as file:
        csv_reader = csv.DictReader(file)
        for row in csv_reader:
            data.append(dict(row))
    return data

def format_row_output(row_num: int, row_data: Dict[str, str]) -> str:
    """
    Format a single row's data for display.
    
    Args:
        row_num: Row number (1-indexed)
        row_data: Dictionary containing row data
        
    Returns:
        Formatted string representation of the row
    """
    output = f"Row {row_num}:\n"
    output += f"  ID: {row_data.get('id', 'N/A')}\n"
    output += f"  First Name: {row_data.get('first_name', 'N/A')}\n"
    output += f"  Last Name: {row_data.get('last_name', 'N/A')}\n"
    output += f"  Email: {row_data.get('email', 'N/A')}\n"
    output += f"  Gender: {row_data.get('gender', 'N/A')}\n"
    output += f"  IP Address: {row_data.get('ip_address', 'N/A')}\n"
    return output

def print_csv_data(file_path: str) -> None:
    """
    Read and print CSV data in a formatted way.
    
    Args:
        file_path: Path to the CSV file
    """
    data = read_csv_file(file_path)
    for idx, row in enumerate(data, start=1):
        print(format_row_output(idx, row))
        print()

# Execute the main function
file_path = '/Volumes/workspace/default/source/MOCK_DATA.csv'
print_csv_data(file_path)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import Row
from typing import List

def read_csv_with_pyspark(file_path: str, header: bool = True, infer_schema: bool = True) -> DataFrame:
    """
    Read CSV file using PySpark and return DataFrame.
    
    Args:
        file_path: Path to the CSV file
        header: Whether the CSV has a header row
        infer_schema: Whether to automatically infer column types
        
    Returns:
        PySpark DataFrame
    """
    df = spark.read.csv(
        file_path,
        header=header,
        inferSchema=infer_schema
    )
    return df

def format_spark_row_output(row_num: int, row: Row) -> str:
    """
    Format a single Spark Row for display.
    
    Args:
        row_num: Row number (1-indexed)
        row: PySpark Row object
        
    Returns:
        Formatted string representation of the row
    """
    output = f"Row {row_num}:\n"
    
    # Define fields to display
    fields = ['id', 'first_name', 'last_name', 'email', 'gender', 'ip_address']
    
    # Check each field individually
    for field in fields:
        try:
            value = row[field]
            field_label = field.replace('_', ' ').title()
            output += f"  {field_label}: {value}\n"
        except (KeyError, ValueError, Exception):
            field_label = field.replace('_', ' ').title()
            output += f"  {field_label}: N/A\n"
    
    return output

def display_dataframe_schema(df: DataFrame) -> None:
    """
    Display the schema of a DataFrame.
    
    Args:
        df: PySpark DataFrame
    """
    print("Schema:")
    df.printSchema()
    print("\n" + "="*80 + "\n")

def print_pyspark_data(file_path: str) -> None:
    """
    Read CSV using PySpark and print data in formatted way.
    
    Args:
        file_path: Path to the CSV file
    """
    # Read the CSV file
    df = read_csv_with_pyspark(file_path)
    
    # Display schema
    display_dataframe_schema(df)
    
    # Collect and print rows
    rows = df.collect()
    for idx, row in enumerate(rows, start=1):
        print(format_spark_row_output(idx, row))
        print()

# Execute the main function
file_path = '/Volumes/workspace/default/source/MOCK_DATA.csv'
print_pyspark_data(file_path)

In [0]:
import unittest
from unittest.mock import patch, MagicMock
from io import StringIO
import csv as csv_module
from pyspark.sql import Row
import tempfile
import os

class TestCSVProcessing(unittest.TestCase):
    """Unit tests for CSV processing functions from Cell 1"""
    
    def setUp(self):
        """Set up test data"""
        self.expected_row_1 = {
            'id': '1',
            'first_name': 'John',
            'last_name': 'Doe',
            'email': 'john@example.com',
            'gender': 'Male',
            'ip_address': '192.168.1.1'
        }
    
    def test_read_csv_file_structure(self):
        """Test reading CSV file returns correct data structure"""
        # Create a temporary CSV file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
            f.write("id,first_name,last_name,email,gender,ip_address\n")
            f.write("1,John,Doe,john@example.com,Male,192.168.1.1\n")
            f.write("2,Jane,Smith,jane@example.com,Female,192.168.1.2\n")
            temp_file_path = f.name
        
        try:
            # Read the file
            result = read_csv_file(temp_file_path)
            
            # Verify it returns a list
            self.assertIsInstance(result, list)
            
            # Verify we have 2 rows
            self.assertEqual(len(result), 2)
            
            # Verify first row data
            self.assertEqual(result[0]['id'], '1')
            self.assertEqual(result[0]['first_name'], 'John')
            self.assertEqual(result[0]['email'], 'john@example.com')
            
            # Verify second row data
            self.assertEqual(result[1]['id'], '2')
            self.assertEqual(result[1]['first_name'], 'Jane')
        finally:
            # Clean up temporary file
            if os.path.exists(temp_file_path):
                os.unlink(temp_file_path)
    
    def test_format_row_output(self):
        """Test formatting a row produces correct output"""
        row_data = self.expected_row_1
        output = format_row_output(1, row_data)
        
        # Verify output contains expected elements
        self.assertIn('Row 1:', output)
        self.assertIn('ID: 1', output)
        self.assertIn('First Name: John', output)
        self.assertIn('Last Name: Doe', output)
        self.assertIn('Email: john@example.com', output)
        self.assertIn('Gender: Male', output)
        self.assertIn('IP Address: 192.168.1.1', output)
    
    def test_format_row_output_with_missing_fields(self):
        """Test formatting handles missing fields gracefully"""
        incomplete_row = {'id': '1', 'first_name': 'John'}
        output = format_row_output(1, incomplete_row)
        
        # Verify missing fields show 'N/A'
        self.assertIn('Last Name: N/A', output)
        self.assertIn('Email: N/A', output)
    
    def test_format_row_output_with_empty_dict(self):
        """Test formatting handles empty dictionary"""
        empty_row = {}
        output = format_row_output(1, empty_row)
        
        # Verify all fields show 'N/A'
        self.assertIn('Row 1:', output)
        self.assertIn('N/A', output)
        
class TestPySparkProcessing(unittest.TestCase):
    """Unit tests for PySpark processing functions from Cell 2"""
    
    def test_format_spark_row_output(self):
        """Test formatting Spark Row produces correct output"""
        # Create an actual Row object
        test_row = Row(id=1, first_name='John', last_name='Doe', 
                      email='john@example.com', gender='Male', ip_address='192.168.1.1')
        
        output = format_spark_row_output(1, test_row)
        
        # Verify output contains expected elements
        self.assertIn('Row 1:', output)
        self.assertIn('Id: 1', output)
        self.assertIn('First Name: John', output)
        self.assertIn('Last Name: Doe', output)
        self.assertIn('Email: john@example.com', output)
        self.assertIn('Gender: Male', output)
        self.assertIn('Ip Address: 192.168.1.1', output)
    
    def test_format_spark_row_with_missing_fields(self):
        """Test formatting Spark Row handles missing fields"""
        # Create a Row with only id field
        test_row = Row(id=1)
        
        output = format_spark_row_output(1, test_row)
        
        # Verify row number is present and missing fields show N/A
        self.assertIn('Row 1:', output)
        self.assertIn('N/A', output)
        # Verify Id is present
        self.assertIn('Id: 1', output)
    
    def test_format_spark_row_with_partial_fields(self):
        """Test formatting Spark Row with some fields present"""
        test_row = Row(id=5, first_name='Alice', email='alice@test.com')
        
        output = format_spark_row_output(1, test_row)
        
        # Verify present fields
        self.assertIn('Id: 5', output)
        self.assertIn('First Name: Alice', output)
        self.assertIn('Email: alice@test.com', output)
        # Verify missing fields show N/A
        self.assertIn('Last Name: N/A', output)
        self.assertIn('Gender: N/A', output)
        self.assertIn('Ip Address: N/A', output)

# Run the tests
if __name__ == '__main__':
    # Create a test suite
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    
    # Add tests
    suite.addTests(loader.loadTestsFromTestCase(TestCSVProcessing))
    suite.addTests(loader.loadTestsFromTestCase(TestPySparkProcessing))
    
    # Run with verbose output
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"Tests run: {result.testsRun}")
    print(f"Failures: {len(result.failures)}")
    print(f"Errors: {len(result.errors)}")
    if result.testsRun > 0:
        success_rate = (result.testsRun - len(result.failures) - len(result.errors)) / result.testsRun * 100
        print(f"Success rate: {success_rate:.1f}%")
    print(f"{'='*70}")